# Building Product Recommendation Systems Using Real-World Retail and Gaming Data

## E-Commerce & Gaming Analytics Pipeline

## Project Overview

###Preprocessing and Data Managemet  

This notebook documents the data preprocessing and management workflow for the E-Commerce & Gaming Analytics Pipeline. It includes loading the datasets, assessing data quality, handling missing values, removing unnecessary variables, merging datasets where appropriate, and preparing the data for subsequent exploratory data analysis (EDA) and machine learning. The cleaned datasets are saved for use in later stages of the project.

### Amazon Product Reviews

This section preprocesses the Amazon Product Reviews dataset by removing unnecessary columns, handling missing values, and preparing the data for subsequent exploratory data analysis (EDA) and machine learning. The cleaned dataset is saved for use in later stages of the project.

In [3]:
import pandas as pd

# Amazon Product Reviews

# Load the raw dataset
amazon = pd.read_csv("Amazon Product Reviews.csv")

# Check initial dataset size
print("Original shape:", amazon.shape)

# Remove columns that are not useful for analysis or modeling
amazon_clean = amazon.drop(columns=[
    'reviews.sourceURLs',
    'reviews.userCity',
    'reviews.userProvince',
    'reviews.username',
    'sizes',
    'upc',
    'ean',
    'reviews.title'
], errors='ignore')

# Remove duplicate rows
amazon_clean = amazon_clean.drop_duplicates()

# Remove rows where the target variable is missing
amazon_clean = amazon_clean.dropna(subset=['reviews.rating'])

# Fill missing numerical values using the column median
# Median is more robust for data with outliers
numeric_cols = amazon_clean.select_dtypes(include=['int64', 'float64']).columns
amazon_clean[numeric_cols] = amazon_clean[numeric_cols].fillna(
    amazon_clean[numeric_cols].median()
)

# Replace missing review text with an empty string
if 'reviews.text' in amazon_clean.columns:
    amazon_clean['reviews.text'] = amazon_clean['reviews.text'].fillna('')

# Fill remaining missing categorical values with "Unknown"
categorical_cols = amazon_clean.select_dtypes(include=['object']).columns
amazon_clean[categorical_cols] = amazon_clean[categorical_cols].fillna('Unknown')


# Check final dataset quality
print("Cleaned shape:", amazon_clean.shape)

print("\nMissing values:")
print(amazon_clean.isnull().sum())

print("\nDuplicate rows:")
print(amazon_clean.duplicated().sum())

# Save the cleaned dataset
amazon_clean.to_csv("amazon_clean.csv", index=False)

print("\nAmazon Product Reviews dataset cleaned and saved successfully.")

Original shape: (1597, 27)
Cleaned shape: (1038, 19)

Missing values:
id                     0
asins                  0
brand                  0
categories             0
colors                 0
dateAdded              0
dateUpdated            0
dimension              0
keys                   0
manufacturer           0
manufacturerNumber     0
name                   0
prices                 0
reviews.date           0
reviews.doRecommend    0
reviews.numHelpful     0
reviews.rating         0
reviews.text           0
weight                 0
dtype: int64

Duplicate rows:
1

Amazon Product Reviews dataset cleaned and saved successfully.


### Steam Dataset

The Steam recommendation data consists of multiple files containing user information, game metadata, and user recommendations. Due to the large size of the recommendations dataset, the data are processed in chunks to reduce memory usage while performing data cleaning and merging.

In [4]:
# Steam Recommendation Dataset

# Load user and game metadata
users = pd.read_csv('users.csv')
games = pd.read_csv('games.csv')

# Standardize column names for consistency
users.columns = users.columns.str.strip().str.lower()
games.columns = games.columns.str.strip().str.lower()

# Process the recommendations dataset in chunks to reduce memory usage
chunk_size = 500000
chunks = []

for chunk in pd.read_csv('recommendations.csv', chunksize=chunk_size, low_memory=False):

    # Standardize column names
    chunk.columns = chunk.columns.str.strip().str.lower()

    # Merge recommendation data with user and game information
    merged_chunk = chunk.merge(users, on='user_id', how='left')
    merged_chunk = merged_chunk.merge(games, on='app_id', how='left')

    # Remove unnecessary columns
    merged_chunk = merged_chunk.drop(
        columns=['user_id', 'review_id', 'tags'],
        errors='ignore'
    )

    # Fill missing numerical values with the column mean
    numeric_cols = merged_chunk.select_dtypes(include=['int64', 'float64']).columns
    merged_chunk[numeric_cols] = merged_chunk[numeric_cols].fillna(
        merged_chunk[numeric_cols].mean()
    )

    # Remove rows with missing target values and convert the target to integer
    if 'is_recommended' in merged_chunk.columns:
        merged_chunk = merged_chunk.dropna(subset=['is_recommended'])
        merged_chunk['is_recommended'] = merged_chunk['is_recommended'].astype(int)

    chunks.append(merged_chunk)

# Combine all processed chunks into a single DataFrame
steam_clean = pd.concat(chunks, ignore_index=True)

# Remove duplicate rows
steam_clean = steam_clean.drop_duplicates()

# Verify the cleaned dataset
print("Shape:", steam_clean.shape)

print("\nMissing values:")
print(steam_clean.isnull().sum())

print("\nDuplicate rows:")
print(steam_clean.duplicated().sum())

# Save the cleaned dataset
steam_clean.to_csv("steam_clean.csv", index=False)

print("Steam Recommendations dataset cleaned and saved successfully.")

Shape: (41135171, 20)

Missing values:
app_id            0
helpful           0
funny             0
date              0
is_recommended    0
hours             0
products          0
reviews           0
title             0
date_release      0
win               0
mac               0
linux             0
rating            0
positive_ratio    0
user_reviews      0
price_final       0
price_original    0
discount          0
steam_deck        0
dtype: int64

Duplicate rows:
0
Steam Recommendations dataset cleaned and saved successfully.


###Video Game Sales

This section preprocesses the Video Game Sales dataset by removing unnecessary columns, handling missing values, and preparing the data for subsequent exploratory data analysis and machine learning. The cleaned dataset is then saved for later use in the modeling pipeline.

In [6]:
import pandas as pd

# Video Game Sales Dataset

# Load the raw dataset
sales = pd.read_csv("video games sales 2019.csv")

# Check initial dataset size
print("Original shape:", sales.shape)

# Remove irrelevant columns
sales_clean = sales.drop(columns=[
    'Name',
    'url',
    'img_url',
    'Last_Update',
    'status',
    'VGChartz_Score',
    'Vgchartzscore'
    'Total_Shipped'
], errors='ignore')

# Remove rows with missing target values
sales_clean = sales_clean.dropna(subset=['Global_Sales'])

# Remove duplicate rows
sales_clean = sales_clean.drop_duplicates()

# Fill missing categorical values
categorical_cols = sales_clean.select_dtypes(include=['object']).columns
sales_clean[categorical_cols] = sales_clean[categorical_cols].fillna('Unknown')

# Fill missing numerical values with the column median
# Median is more robust for sales data with outliers
numeric_cols = sales_clean.select_dtypes(include=['int64', 'float64']).columns
sales_clean[numeric_cols] = sales_clean[numeric_cols].fillna(
    sales_clean[numeric_cols].median()
)

# Check final dataset quality
print("Cleaned shape:", sales_clean.shape)

print("\nMissing values:")
print(sales_clean.isnull().sum())

print("\nDuplicate rows:")
print(sales_clean.duplicated().sum())

# Save the cleaned dataset
sales_clean.to_csv("sales_clean.csv", index=False)

print("\nVideo Game Sales dataset cleaned and saved successfully.")

Original shape: (55792, 23)
Cleaned shape: (19415, 17)

Missing values:
Rank                 0
basename             0
Genre                0
ESRB_Rating          0
Platform             0
Publisher            0
Developer            0
Critic_Score         0
User_Score           0
Total_Shipped    19415
Global_Sales         0
NA_Sales             0
PAL_Sales            0
JP_Sales             0
Other_Sales          0
Year                 0
Vgchartzscore        0
dtype: int64

Duplicate rows:
0

Video Game Sales dataset cleaned and saved successfully.


## Save Cleaned Data

After preprocessing, the cleaned datasets are converted from CSV to Parquet format. Parquet provides efficient storage, faster read and write performance, and reduced file size, making it well suited for large datasets and subsequent exploratory data analysis (EDA) and machine learning.


# Convert Steam dataset from CSV to Parquet

In [7]:

import pyarrow as pa
import pyarrow.parquet as pq

# Open parquet writer
writer = None
chunksize = 500000

for chunk in pd.read_csv("steam_clean.csv", chunksize=chunksize, low_memory=False):
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter("steam_clean.parquet", table.schema)
    writer.write_table(table)

if writer:
    writer.close()

print("Steam CSV written to Parquet in streaming mode")


Steam CSV written to Parquet in streaming mode


# Convert Amazon and Video Game Sales datasets to Parquet

In [8]:

# Amazon & Sales
amazon_clean = pd.read_csv("amazon_clean.csv", low_memory=False)
sales_clean = pd.read_csv("sales_clean.csv", low_memory=False)

amazon_clean.to_parquet("amazon_clean.parquet", index=False)
sales_clean.to_parquet("sales_clean.parquet", index=False)

print(" Amazon & Sales converted to Parquet")


 Amazon & Sales converted to Parquet


The cleaned Parquet files are loaded for exploratory data analysis and machine learning.

In [9]:
import pandas as pd

amazon_clean = pd.read_parquet("amazon_clean.parquet")
sales_clean = pd.read_parquet("sales_clean.parquet")
steam_clean = pd.read_parquet("steam_clean.parquet")



print("All cleaned datasets loaded successfully.")

All cleaned datasets loaded successfully.
